# Trabalho Prático 1: O Analista de Dados
## ADS308 - Inteligência Artificial

**Prof.:** João Vítor Rodrigues de Vasconcelos

**Alunos:** Bernardo Motta e Diego Araújo

**Curso:** Engenharia de Computação / Análise e Desenvolvimento de Sistemas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Importação das Bibliotecas

Importamos as bibliotecas necessárias para manipulação de dados (Pandas, Numpy), visualização (Matplotlib, Seaborn) e cálculo estatístico (Scipy).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style='whitegrid')

## 3.1. Carregamento e Inspeção Inicial

Nesta etapa, carregamos o dataset bruto utilizando a biblioteca Pandas e realizamos a inspeção inicial para entender a estrutura dos dados: formato das colunas, tipos de dados e presença de valores nulos.

In [ ]:
# Caminho obrigatório do arquivo
CAMINHO_CSV = '/content/drive/MyDrive/Colab Notebooks/dataset_vendas_sujo_26356.csv'

# Carregando o dataset
df = pd.read_csv(CAMINHO_CSV)

# Verificar se o arquivo carregado é realmente o dataset de vendas
colunas_esperadas = ['ID_Pedido', 'Data_Venda', 'Categoria', 'Preco_Unitario', 'Quantidade', 'Idade_Cliente']
colunas_presentes = list(df.columns)

print(f'Colunas encontradas no arquivo: {colunas_presentes}')

if not all(col in colunas_presentes for col in colunas_esperadas):
    print('\n⚠️  ATENÇÃO: O arquivo carregado NÃO contém as colunas do dataset de vendas.')
    print('   O arquivo no caminho especificado parece ser outro dataset (ex: imóveis).')
    print('   Por favor, substitua o arquivo no Google Drive pelo dataset_vendas_sujo.csv correto.')
    print(f'   Colunas esperadas: {colunas_esperadas}')
    print(f'   Colunas encontradas: {colunas_presentes}')
    raise ValueError('Dataset incorreto. Substitua o arquivo no caminho indicado pelo dataset de vendas.')
else:
    print('\n✅ Dataset de vendas carregado com sucesso!')

In [ ]:
# Primeiras e últimas linhas
print('--- Primeiras 5 linhas ---')
display(df.head())

print('\n--- Últimas 5 linhas ---')
display(df.tail())

In [ ]:
# Resumo estrutural: tipos de dados e contagem total de registros
print('--- Resumo Estrutural do Dataset ---')
print(f'Total de registros: {len(df)}')
print(f'Total de colunas: {len(df.columns)}')
print()
df.info()

In [ ]:
# Contagem de valores nulos por coluna
print('--- Contagem de Valores Nulos por Coluna ---')
print(df.isnull().sum())
print(f'\nTotal geral de valores nulos: {df.isnull().sum().sum()}')

### Observações da Inspeção Inicial

Ao inspecionar o dataset, identificamos os seguintes problemas que precisam ser tratados na etapa de limpeza:

* A coluna `Preco_Unitario` contém valores com prefixo "R$" e vírgula como separador decimal (ex: `"R$ 155,32"`), sendo lida como texto (object) ao invés de número.
* A coluna `Data_Venda` possui entradas inválidas como `"Sem Data"`.
* A coluna `Categoria` apresenta erros de digitação e falta de padronização (ex: `"csa"`, `"roupa"`, `"ELETRONICOS"`, `"Eletrônico "`).
* Existem valores nulos em diversas colunas (`Preco_Unitario`, `Quantidade`, `Idade_Cliente`).
* Há indícios de outliers extremos nos preços (valores como `99999.99` e `-50.0`) e nas idades (valores negativos e acima de 100).

## 3.2. Limpeza e Tratamento de Dados (Data Cleaning)

Nesta etapa, realizamos a limpeza do dataset para garantir a integridade das análises futuras. As ações e justificativas são detalhadas abaixo.

### 3.2.1. Tipagem: Conversão da coluna de Preço

A coluna `Preco_Unitario` possui valores mistos: alguns são numéricos puros e outros estão formatados como string com prefixo `"R$ "` e vírgula decimal (ex: `"R$ 155,32"`). Removemos o prefixo, substituímos a vírgula por ponto e convertemos para `float`.

In [ ]:
# Converter Preco_Unitario: remover 'R$ ', trocar vírgula por ponto, converter para float
df['Preco_Unitario'] = (
    df['Preco_Unitario']
    .astype(str)
    .str.replace(r'R\$\s*', '', regex=True)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
df['Preco_Unitario'] = pd.to_numeric(df['Preco_Unitario'], errors='coerce')

# Garantir que Quantidade e Idade sejam numéricas
df['Quantidade'] = pd.to_numeric(df['Quantidade'], errors='coerce')
df['Idade_Cliente'] = pd.to_numeric(df['Idade_Cliente'], errors='coerce')

print('Tipos após conversão:')
print(df.dtypes)

### 3.2.2. Tipagem: Conversão da coluna de Data

A coluna `Data_Venda` contém entradas inválidas como `"Sem Data"`. Convertemos para o tipo `datetime`, forçando valores inválidos a se tornarem `NaT`, e em seguida removemos essas linhas.

**Justificativa da remoção:** Sem uma data válida, a venda não pode ser posicionada no tempo, o que inviabiliza análises temporais. Como são poucas linhas, a remoção não impacta significativamente o volume de dados.

In [ ]:
# Converter Data_Venda para datetime (entradas inválidas como 'Sem Data' viram NaT)
df['Data_Venda'] = pd.to_datetime(df['Data_Venda'], errors='coerce')

n_datas_invalidas = df['Data_Venda'].isna().sum()
print(f'Linhas com data inválida (NaT): {n_datas_invalidas}')

# Remover linhas com data inválida
df = df.dropna(subset=['Data_Venda']).copy()
print(f'Registros restantes após remoção de datas inválidas: {len(df)}')

### 3.2.3. Padronização de Categorias (Corrigindo erros de digitação)

A coluna `Categoria` apresenta variações de escrita para a mesma categoria. Padronizamos todas para o formato correto. As regras aplicadas foram:

* `"csa"` → `"Casa"`
* `"roupa"` → `"Roupas"`
* `"eletronico"`, `"eletrônico"`, `"eletronicos"` → `"Eletrônicos"`
* `"CASA"`, `"ROUPAS"`, `"ELETRONICOS"` → formas padronizadas via mapeamento

In [ ]:
# Mostrar valores únicos antes da limpeza
print('Categorias ANTES da padronização:')
print(df['Categoria'].value_counts())
print()

# Normalizar: remover espaços extras e converter para minúsculas
df['Categoria'] = df['Categoria'].astype(str).str.strip().str.lower()

# Mapa de correções completo (tudo em minúsculas como chave)
correcoes_categoria = {
    'alimentos': 'Alimentos',
    'casa': 'Casa',
    'csa': 'Casa',
    'roupas': 'Roupas',
    'roupa': 'Roupas',
    'eletrônicos': 'Eletrônicos',
    'eletronicos': 'Eletrônicos',
    'eletronico': 'Eletrônicos',
    'eletrônico': 'Eletrônicos',
    'beleza': 'Beleza',
}
df['Categoria'] = df['Categoria'].map(correcoes_categoria).fillna('Outros')

print('Categorias APÓS a padronização:')
print(df['Categoria'].value_counts())

### 3.2.4. Dados Faltantes: Imputação pela Mediana

Optamos por preencher os valores nulos das colunas numéricas utilizando a **mediana** ao invés da média.

**Justificativa:** A mediana é uma medida de tendência central robusta, ou seja, não é distorcida por valores extremos (outliers). Como o dataset contém preços absurdos (R$ 99.999,99) e idades impossíveis, a mediana representa melhor o valor típico da distribuição.

In [ ]:
print('Valores nulos ANTES da imputação:')
print(df[['Preco_Unitario', 'Quantidade', 'Idade_Cliente']].isnull().sum())

# Imputação pela mediana
for col in ['Preco_Unitario', 'Quantidade', 'Idade_Cliente']:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)
    print(f'  {col}: mediana usada = {mediana}')

print('\nValores nulos APÓS a imputação:')
print(df[['Preco_Unitario', 'Quantidade', 'Idade_Cliente']].isnull().sum())

### 3.2.5. Outliers: Identificação e Remoção pelo Z-Score

Utilizamos o método do **Z-Score** para identificar valores atípicos na coluna `Preco_Unitario`. A fórmula é:

$$Z = \frac{x - \mu}{\sigma}$$

Onde $\mu$ é a média e $\sigma$ é o desvio padrão.

Removemos registros com $|Z| \geq 3$, ou seja, preços que estão a mais de 3 desvios padrões da média. Isso elimina valores anômalos como itens custando R$ 99.999,99 ou preços negativos (-50).

Também tratamos idades inválidas (negativas ou acima de 100 anos) como uma camada adicional de limpeza.

In [ ]:
print(f'Registros antes da remoção de outliers: {len(df)}')

# Z-Score na coluna Preco_Unitario
media_preco = df['Preco_Unitario'].mean()
std_preco = df['Preco_Unitario'].std()
df['Z_Preco'] = (df['Preco_Unitario'] - media_preco) / std_preco

outliers_preco = (df['Z_Preco'].abs() >= 3).sum()
print(f'Outliers de preço detectados (|Z| >= 3): {outliers_preco}')

# Exemplos de outliers removidos
if outliers_preco > 0:
    print('Exemplos de outliers de preço:')
    display(df[df['Z_Preco'].abs() >= 3][['ID_Pedido', 'Preco_Unitario', 'Z_Preco']].head(10))

df = df[df['Z_Preco'].abs() < 3].copy()
df = df.drop(columns=['Z_Preco'])  # remover coluna auxiliar

# Tratar idades inválidas (negativas ou > 100)
idades_invalidas = ((df['Idade_Cliente'] < 0) | (df['Idade_Cliente'] > 100)).sum()
print(f'Idades inválidas detectadas (< 0 ou > 100): {idades_invalidas}')
df = df[(df['Idade_Cliente'] >= 0) & (df['Idade_Cliente'] <= 100)].copy()

print(f'\nRegistros finais após remoção de outliers: {len(df)}')

In [ ]:
# Criar colunas auxiliares para a EDA
df['Mes'] = df['Data_Venda'].dt.month
df['Total_Venda'] = df['Preco_Unitario'] * df['Quantidade']

print('--- Dataset Limpo: Resumo Final ---')
df.info()
print()
display(df.describe())

## 3.3. Análise Exploratória (EDA)

Com o dataset limpo, utilizamos Matplotlib e Seaborn para responder visualmente a três perguntas de negócio:

1. **Qual a distribuição das vendas ao longo dos meses?** (Gráfico de Barras)
2. **Existe correlação entre o preço do produto e a quantidade vendida?** (Gráfico de Dispersão)
3. **Como as vendas se dividem por categoria de produto?** (Gráfico de Barras Horizontais)

### Pergunta 1: Qual a distribuição do faturamento total ao longo dos meses de 2023?

In [ ]:
plt.figure(figsize=(12, 5))

vendas_por_mes = df.groupby('Mes')['Total_Venda'].sum().reindex(range(1, 13), fill_value=0)

meses_labels = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
                'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']

plt.bar(range(1, 13), vendas_por_mes.values, color=sns.color_palette('Blues_d', 12))
plt.title('Faturamento Total ao Longo dos Meses (2023)', fontsize=14, fontweight='bold')
plt.xlabel('Mês', fontsize=12)
plt.ylabel('Faturamento Total (R$)', fontsize=12)
plt.xticks(range(1, 13), meses_labels)
plt.tight_layout()
plt.show()

### Pergunta 2: Existe correlação entre o preço do produto e a quantidade vendida?

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['Preco_Unitario'], df['Quantidade'], alpha=0.3, color='coral', edgecolors='none')

# Calcular e exibir a correlação de Pearson
corr = df['Preco_Unitario'].corr(df['Quantidade'])
plt.title(f'Preço Unitário vs Quantidade Vendida (Correlação: {corr:.4f})', fontsize=14, fontweight='bold')
plt.xlabel('Preço Unitário (R$)', fontsize=12)
plt.ylabel('Quantidade (Unidades)', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Coeficiente de correlação de Pearson: {corr:.4f}')

### Pergunta 3: Como as vendas se dividem por categoria de produto?

In [ ]:
plt.figure(figsize=(10, 6))
vendas_categoria = df.groupby('Categoria')['Total_Venda'].sum().sort_values(ascending=True)

cores = sns.color_palette('viridis', len(vendas_categoria))
plt.barh(vendas_categoria.index, vendas_categoria.values, color=cores)
plt.title('Faturamento Total por Categoria de Produto', fontsize=14, fontweight='bold')
plt.xlabel('Faturamento Total (R$)', fontsize=12)
plt.ylabel('Categoria', fontsize=12)
plt.tight_layout()
plt.show()

print('\nFaturamento por categoria:')
for cat, valor in vendas_categoria.sort_values(ascending=False).items():
    print(f'  {cat}: R$ {valor:,.2f}')

## 3.4. Conclusões e Insights

Com base na Análise Exploratória de Dados (EDA) realizada sobre o dataset de vendas de 2023, consolidamos os seguintes insights e recomendações:

---

**Insight 1 — Ausência de Sensibilidade ao Preço (Oportunidade de Upsell)**

A correlação entre preço unitário e quantidade vendida é praticamente nula (próxima de 0). Isso indica que os clientes não reduzem a quantidade comprada quando o produto é mais caro. A gerência pode explorar estratégias de upsell (oferecer versões premium dos produtos) e kits promocionais sem medo de perder volume de vendas.

**Insight 2 — Distribuição Relativamente Estável ao Longo do Ano**

O faturamento mensal apresenta uma distribuição relativamente uniforme ao longo de 2023, sem grandes picos sazonais. Isso sugere que a rede de varejo possui uma base de clientes recorrente. Meses com leve queda poderiam ser alvo de campanhas promocionais para equilibrar ainda mais o faturamento.

**Insight 3 — Categorias Líderes em Faturamento**

As categorias Beleza, Alimentos e Roupas competem pelo topo do faturamento, enquanto Casa e Eletrônicos também possuem participação relevante. Recomenda-se garantir estoque robusto para as categorias líderes e investigar oportunidades de crescimento nas categorias de menor desempenho relativo.

**Insight 4 — Necessidade Urgente de Governança de Dados**

Durante a limpeza, identificamos problemas graves na base de dados original: preços irreais (R$ 99.999,99), preços negativos (-R$ 50,00), idades impossíveis (203, 227, -5 anos), datas inválidas ("Sem Data") e erros de digitação nas categorias ("csa", "roupa", "Eletrônico ", "ELETRONICOS"). Recomenda-se fortemente que a equipe de TI implemente validações no sistema de cadastro para evitar a poluição do banco de dados.

**Insight 5 — Perfil de Cliente Diversificado**

A faixa etária dos clientes (após limpeza) varia de 18 a 74 anos, com mediana por volta dos 45 anos. Isso indica uma base de clientes diversificada em termos de idade. Estratégias de marketing segmentadas por faixa etária podem ser desenvolvidas para aumentar a conversão em cada grupo.